<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Pretrained detectors

**[Pretrained detectors](https://learning.nextia-ai.com/courses/vision/m04/pretrained-detectors/)** · Vision AI: See, Count and Inspect · Module 4, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** run a pretrained detector on pallet photos, read the classes it knows, and judge a detector with average precision (AP) by object size.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells and the detector's weights (74 MB). No account and no key. |
| **You should know** | Boxes, IoU and non-maximum suppression, from [Boxes and IoU](https://learning.nextia-ai.com/courses/vision/m04/boxes-and-iou/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1 and matplotlib 3.11.2 (PyTorch 2.15.0 and timm 1.0.31 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m04/pretrained-detectors/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M04-L02-pretrained-detectors/pretrained-detectors-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, torch 2.15.0, torchvision 0.29.2, timm 1.0.31, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time. PyTorch is large (about 200 MB to download): the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2 torch==2.15.0 torchvision==0.29.2 timm==1.0.31
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)
import torch; print("torch", torch.__version__)
import torchvision; print("torchvision", torchvision.__version__)
import timm; print("timm", timm.__version__)

## Setup: get the Larkfield images

Run the next cell. It downloads the image sets that this lesson needs (about 6 MB, as zip files) into a folder `larkfield-images`, with the course's helper `compare.py`, checks each file's checksum and unpacks it. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-images` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the catalogue photos are cut from Open Images photos (CC BY 2.0: the author of each photo is in `data/catalogue/manifest.csv`), the returned candles are photos of the VisA inspection set (CC BY 4.0), and the pallets are drawn by our code. Read `data/dataset-card.md` for the details.

In [ ]:
# Setup: download the image sets into larkfield-images/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "archives/pallets.zip": "cec4b211fa4e434936fc0c1980accfc74392f1ced9de6c04b5421b9ec6af56e3",
    "archives/extras.zip": "f121137bc765866c25929a4233c557ba72b64eea0c806070cc04a86c28ec27ec",
    "compare.py": "e9794acacbce00c240906cc8b3e2256b8505ff4a8074b7507a8b159cd39b4826",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-images").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-images and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in larkfield-images/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The pallets and their true boxes

The 30 test pallets are drawn by our code, so we know every true box. A box behind another box is not counted: the count is the boxes you can see.

In [ ]:
pallets = json.load(open("data/pallets/test.json"))
images = {im["scene_id"]: im for im in pallets["images"]}
by_id = {im["id"]: im["scene_id"] for im in pallets["images"]}
truth = {}
for a in pallets["annotations"]:
    truth.setdefault(by_id[a["image_id"]], []).append(a["bbox"])
print(len(images), "photos,", sum(len(v) for v in truth.values()), "true boxes")

## 2. Run a pretrained detector

torchvision's Faster R-CNN with a MobileNetV3 backbone was trained on COCO: 80 classes of everyday objects, such as person, car, clock and keyboard. COCO has no class "cardboard box".

> **Predict.** The detector has never seen the class "box". What will it say about a pallet of boxes?

In [ ]:
import torch
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn, FasterRCNN_MobileNet_V3_Large_FPN_Weights
torch.set_num_threads(4)
weights = FasterRCNN_MobileNet_V3_Large_FPN_Weights.COCO_V1
detector = fasterrcnn_mobilenet_v3_large_fpn(weights=weights, box_score_thresh=0.05).eval()
names = weights.meta["categories"]
prepare = weights.transforms()

def detect(sid):
    """The detector's boxes on one photo: class, score and [x, y, w, h]."""
    x = prepare(Image.open("data/" + images[sid]["file_name"]).convert("RGB"))
    with torch.no_grad():
        o = detector([x])[0]
    return [(names[l], round(s, 2), [round(v, 1) for v in (b[0], b[1], b[2] - b[0], b[3] - b[1])])
            for b, s, l in zip(o["boxes"].tolist(), o["scores"].tolist(), o["labels"].tolist())]

for sid in ("S-061", "S-063", "S-085"):
    found = [f"{c} {s}" for c, s, _ in detect(sid) if s >= 0.5]
    print(sid, images[sid]["crowding"], images[sid]["count"], "boxes ->", found or "nothing at score 0.5")

The detector sees a clock or a train where we see boxes. It reports only the classes that it learned.

## 3. The recorded boxes of two COCO detectors on all 30 photos

The course team ran Faster R-CNN and RT-DETR (Apache-2.0) on the 30 test photos once and saved every box with a score of at least 0.05. RT-DETR needs the Transformers package, so we replay its boxes here.

In [ ]:
from collections import Counter
recorded = {name: json.load(open(f"data/predictions/pallets_{name}.json")) for name in ("fasterrcnn_coco", "rtdetr_coco")}
for name, P in recorded.items():
    print(name, dict(Counter(p["class"] for p in P if p["score"] >= 0.5)))

## 4. Average precision, by object size

Average precision (AP) at IoU 0.5 puts every box of every photo in one list, best score first, and measures how precision falls as recall rises. 1.0 is perfect. `cp.average_precision` does it; we treat every class as "box", so a stop sign on a box counts as a box.

COCO sorts objects by area: small (under 32 × 32 pixels), medium (up to 96 × 96) and large.

In [ ]:
gts = truth
for name, P in recorded.items():
    T = [(p["scene_id"], p["score"], p["box"]) for p in P]
    ap = cp.average_precision(gts, T, 0.5)
    by_size = {k: cp.average_precision(gts, T, 0.5, rng) for k, rng in cp.SIZES.items()}
    print(f"{name:16s} AP50 {ap:.3f} | " + " ".join(f"{k} {v:.3f}" for k, v in by_size.items()))
sizes = Counter(next(k for k, (lo, hi) in cp.SIZES.items() if lo <= b[2] * b[3] < hi) for bs in gts.values() for b in bs)
print("true boxes by size:", dict(sizes))

> **Check.** AP50 is 0.0 for Faster R-CNN and 0.013 for RT-DETR: almost nothing. There are no small boxes, so AP for "small" is `nan` (not a number): there is nothing to find.

## 5. Your turn: AP on medium boxes

> **Your turn.** Put RT-DETR's AP at IoU 0.5 on the **medium** boxes only in `ap_medium` (use `cp.average_precision` with `cp.SIZES["medium"]`).

In [ ]:
ap_medium = ...

In [ ]:
expect('ap_medium', ap_medium, 0.042, tolerance=0.01)

## 6. Failure case: a good detector for the wrong classes

RT-DETR is fast and accurate on COCO. On Larkfield's pallets, it calls a top row of boxes "stop sign" and a whole pallet "keyboard". A pretrained detector is only as good as the overlap between its classes and yours. The next lesson teaches it the class "box" with 40 labelled photos.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/vision/m04/pretrained-detectors/) explains AP step by step, the YOLO licence question, and has the knowledge checks.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Fine-tune a detector on a few images](https://learning.nextia-ai.com/courses/vision/m04/fine-tune-a-detector-on-a-few-images/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m04/pretrained-detectors/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are open photos (each with its author and licence in the manifests) or images made by our code; no photo shows a real person.